# Batch 5D — Revision 5 Frontier API Baseline

I use this notebook to run the pre-committed 100-row Revision 5 comparison between frozen ComplianceGPT outputs and the stable Gemini API model `gemini-3.5-flash`. The run preserves the exact questions, ordered evidence windows, free-form prompt/parser, ASK policy, and offline verifier specified in the registered design.

This notebook is configured to request an A100 GPU in Colab. Before executing the run, confirm **Runtime → Change runtime type → A100 GPU**. Gemini generation is performed through the hosted API, while the A100 runtime keeps this notebook consistent with the project's GPU-first execution policy and avoids a CPU-only local environment.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Before running

To reproduce the run, add a Paid Tier 1 API key to Colab's **Secrets** panel (key icon) as `GEMINI_API_KEY`, enable notebook access, and confirm that the key belongs to the intended billed Google project. The notebook never prints or saves the key.

In [2]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/Wenjia1215/ComplianceGPT.git'
SOURCE_COMMIT = 'b7e10b0993147378ade09853265ac8af1214a729'
REPO_DIR = Path('/content/ComplianceGPT_batch5d')
OUTPUT_DIR = Path('/content/drive/MyDrive/rq2_frontier_baseline_rev5_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / '.git').is_dir():
    dirty = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError(
            f'{REPO_DIR} contains local changes. Start a fresh runtime or move that directory.'
        )
else:
    subprocess.run(
        ['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(['git', 'fetch', 'origin', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
checked_out = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
assert checked_out == SOURCE_COMMIT, (checked_out, SOURCE_COMMIT)
print('Pinned Batch 5D source:', checked_out)
print('Persistent result directory:', OUTPUT_DIR)

Pinned Batch 5D source: b7e10b0993147378ade09853265ac8af1214a729
Persistent result directory: /content/drive/MyDrive/rq2_frontier_baseline_rev5_v1


In [3]:
import json
from importlib.metadata import version

runtime_manifest = OUTPUT_DIR / 'manifests/api_runtime.json'
packages = ['numpy==2.0.2', 'PyYAML==6.0.3']
if runtime_manifest.is_file():
    prior_runtime = json.loads(runtime_manifest.read_text(encoding='utf-8'))
    prior_google_genai = str(
        prior_runtime.get('google_genai_python', '') or ''
    ).strip()
    if prior_google_genai:
        packages.append(f'google-genai=={prior_google_genai}')
        print('Resuming with recorded Google Gen AI SDK:', prior_google_genai)
    else:
        packages.append('google-genai>=2.25.0,<3')
else:
    packages.append('google-genai>=2.25.0,<3')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True
)
print('Google Gen AI SDK:', version('google-genai'))

Google Gen AI SDK: 2.26.0


In [4]:
import os
from google.colab import userdata

try:
    api_key = str(userdata.get('GEMINI_API_KEY') or '').strip()
except Exception as exc:
    raise RuntimeError(
        'Add GEMINI_API_KEY in Colab Secrets (key icon), enable notebook access, then rerun this cell.'
    ) from exc
if not api_key:
    raise RuntimeError('GEMINI_API_KEY is empty in Colab Secrets.')
if any(character.isspace() for character in api_key):
    raise RuntimeError(
        'GEMINI_API_KEY contains whitespace. Store only the key value, without an export statement.'
    )
lowered_key = api_key.lower()
if lowered_key.startswith(('bearer ', 'gemini_api_key=', 'google_api_key=')):
    raise RuntimeError(
        'GEMINI_API_KEY must contain only the key value, without Bearer or a variable name.'
    )
if api_key.startswith('AQ.'):
    credential_family = 'authorization key (AQ.)'
elif api_key.startswith('AIza'):
    credential_family = 'standard API key (AIza)'
else:
    credential_family = 'unrecognized key prefix'
os.environ['GEMINI_API_KEY'] = api_key
del api_key, lowered_key
print(
    'GEMINI_API_KEY loaded into process memory; its value was not displayed or written. '
    f'Credential family: {credential_family}.'
)

GEMINI_API_KEY loaded into process memory; its value was not displayed or written. Credential family: authorization key (AQ.).


In [5]:
test_environment = os.environ.copy()
test_environment['PYTHONPATH'] = str(REPO_DIR / 'src')
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_directory = REPO_DIR / 'tests'
required_tests = (
    'test_frontier_api_answerer.py',
    'test_rq2_frontier_baseline.py',
    'test_rq2_frontier_baseline_rev5.py',
)
for test_name in required_tests:
    if not (test_directory / test_name).is_file():
        raise FileNotFoundError(f'Batch 5D test file is missing: {test_directory / test_name}')
test_command = [
    sys.executable, '-m', 'unittest', 'discover',
    '-s', str(test_directory), '-p', 'test_*frontier*.py', '-v',
]
test_result = subprocess.run(
    test_command, cwd=REPO_DIR, env=test_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(test_result.stdout, end='')
if test_result.returncode != 0:
    raise RuntimeError(
        f'Batch 5D offline tests failed (exit {test_result.returncode}). '
        'See the unittest output above for the underlying error.'
    )
print('Batch 5D offline tests passed.')

test_existing_log_locks_settings_and_model (test_frontier_api_answerer.FrontierAPIAnswererTest.test_existing_log_locks_settings_and_model) ... ok
test_invalid_thinking_level_is_forbidden (test_frontier_api_answerer.FrontierAPIAnswererTest.test_invalid_thinking_level_is_forbidden) ... ok
test_model_change_is_logged_then_rejected (test_frontier_api_answerer.FrontierAPIAnswererTest.test_model_change_is_logged_then_rejected) ... ok
test_parse_retry_is_logged_and_preserves_gemini_roles (test_frontier_api_answerer.FrontierAPIAnswererTest.test_parse_retry_is_logged_and_preserves_gemini_roles) ... ok
test_usage_summary_tolerates_missing_fields (test_frontier_api_answerer.FrontierAPIAnswererTest.test_usage_summary_tolerates_missing_fields) ... ok
test_valid_response_reuses_frozen_prompt_and_normalizer (test_frontier_api_answerer.FrontierAPIAnswererTest.test_valid_response_reuses_frozen_prompt_and_normalizer) ... ok
test_activity_detects_call_log_without_completed_csv_row (test_rq2_frontier_base

In [6]:
import hashlib
from IPython.display import Markdown, display

FROZEN_ARCHIVE_SHA256 = '56a70db6eca0420df6affbe63c859418ac423b54d80d3d1eae7bf785f0f63328'
PRECOMMITMENT_SHA256 = 'b0cc78debb25c1025f05622348925383b7a19503751680e1a6a0b3ddefa34504'
frozen_archive = (
    REPO_DIR / 'experiments/answerer_comparison/rq2_matched/results_v3/compliancegpt_rq2_matched_v3.zip'
)
precommitment = (
    REPO_DIR / 'experiments/answerer_comparison/rq2_frontier_baseline_rev5/precommitment.json'
)
actual_archive_sha256 = hashlib.sha256(frozen_archive.read_bytes()).hexdigest()
actual_precommitment_sha256 = hashlib.sha256(precommitment.read_bytes()).hexdigest()
assert actual_archive_sha256 == FROZEN_ARCHIVE_SHA256
assert actual_precommitment_sha256 == PRECOMMITMENT_SHA256
runner = (
    REPO_DIR / 'experiments/answerer_comparison/rq2_frontier_baseline_rev5/run_frontier_baseline_rev5.py'
)
prepare_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--prepare-only',
]
subprocess.run(prepare_command, cwd=REPO_DIR, check=True)
contexts_path = OUTPUT_DIR / 'contexts/rev5_prepared_contexts.jsonl'
with contexts_path.open('r', encoding='utf-8') as handle:
    prepared_rows = sum(1 for line in handle if line.strip())
assert prepared_rows == 100, f'Expected 100 prepared contexts, found {prepared_rows}'
preflight_summary = json.loads(
    (OUTPUT_DIR / 'preflight/summary.json').read_text(encoding='utf-8')
)
assert preflight_summary.get('scope') == 'gold-free frozen-context integrity audit; no API call'
assert preflight_summary.get('api_calls_performed') == 0
print('Frozen archive verified:', actual_archive_sha256)
print('Pre-commitment verified:', actual_precommitment_sha256)
print(f'Prepared-context audit passed: {prepared_rows} Revision 5 contexts; no API call made.')
display(Markdown((OUTPUT_DIR / 'PREPARED_CONTEXTS.md').read_text(encoding='utf-8')))

Frozen archive verified: 56a70db6eca0420df6affbe63c859418ac423b54d80d3d1eae7bf785f0f63328
Pre-commitment verified: b0cc78debb25c1025f05622348925383b7a19503751680e1a6a0b3ddefa34504
Prepared-context audit passed: 100 Revision 5 contexts; no API call made.


# Batch 5D Prepared-Context Audit

This is a gold-free integrity check, not a completed frontier-model result.

- Result identity: `rq2_frontier_baseline_rev5_v1`
- Frozen Revision 5 contexts: 100
- Distinct context hashes: 100
- Distinct evidence-window hashes: 100
- Context SHA-256: `b47e58c17ce6d0aca1d8519941f52c3c21e393d9187fcf113de7bfec68cfc473`
- Pre-commitment SHA-256: `b0cc78debb25c1025f05622348925383b7a19503751680e1a6a0b3ddefa34504`
- API calls performed: no


## Run the registered API comparison

The registered comparison contains 100 rows. The runner checkpoints every returned API response and validated row. It waits and resumes automatically after a transient per-minute 429, while authentication errors, daily-quota failures, and other errors stop the run for inspection. If the runtime disconnects, reconnecting and selecting **Runtime → Run all** safely resumes from the validated checkpoints.

In [7]:
from collections import deque
import math
import re
import time

def per_minute_quota_wait_seconds(output_text):
    if '429 RESOURCE_EXHAUSTED' not in output_text:
        return None
    if 'PerMinute' not in output_text and 'requests per minute' not in output_text.lower():
        return None
    matches = []
    for pattern in (r'Please retry in\s+([0-9]+(?:\.[0-9]+)?)s',):
        matches.extend(float(value) for value in re.findall(pattern, output_text))
    server_delay = max(matches, default=60.0)
    return max(65, math.ceil(server_delay + 5.0))

run_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
]
run_environment = os.environ.copy()
maximum_quota_restarts = 50
quota_restarts = 0
failure_log = OUTPUT_DIR / 'runner_failure_tail.log'
print('Starting the 100-row Revision 5 Gemini 3.5 Flash API baseline.')
print('Transient per-minute 429 responses will wait and resume automatically.')
print('After a disconnect, reconnect and Run all; validated rows are skipped.')
while True:
    tail = deque(maxlen=400)
    process = subprocess.Popen(
        run_command, cwd=REPO_DIR, env=run_environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        tail.append(line)
    return_code = process.wait()
    if return_code == 0:
        break
    failure_output = ''.join(tail)
    failure_log.write_text(failure_output, encoding='utf-8')
    if '401 UNAUTHENTICATED' in failure_output:
        raise RuntimeError(
            'Gemini rejected GEMINI_API_KEY. Confirm that the current Colab account can access '
            'the intended Secret, that notebook access is enabled, and that the key is active '
            'in the intended paid Google AI Studio project. This authentication error is not retryable.'
        )
    wait_seconds = per_minute_quota_wait_seconds(failure_output)
    if wait_seconds is None or quota_restarts >= maximum_quota_restarts:
        print(f'Batch 5D failed. The final runner output is saved at {failure_log}.')
        raise subprocess.CalledProcessError(return_code, run_command)
    quota_restarts += 1
    print(
        f'Per-minute Gemini quota reached; waiting {wait_seconds}s before resumable restart '
        f'{quota_restarts}/{maximum_quota_restarts}. Completed rows remain checkpointed.'
    )
    time.sleep(wait_seconds)
if failure_log.exists():
    failure_log.unlink()
print('Batch 5D runner completed successfully.')

Starting the 100-row Revision 5 Gemini 3.5 Flash API baseline.
Transient per-minute 429 responses will wait and resume automatically.
After a disconnect, reconnect and Run all; validated rows are skipped.
Frozen Batch 5D inputs verified: 100 Revision 5 contexts.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
[generative_frontier_api] 1/100 completed
[generative_frontier_api] 2/100 completed
[generative_frontier_api] 3/100 completed
[generative_frontier_api] 4/100 completed
[generative_frontier_api] 5/100 completed
[generative_frontier_api] 6/100 completed
[generative_frontier_api] 7/100 completed
[generative_frontier_api] 8/100 completed
[generative_frontier_api] 9/100 completed
[generative_frontier_api] 10/100 completed
[generative_frontier_a

In [8]:
summary_path = OUTPUT_DIR / 'summary.json'
api_manifest_path = OUTPUT_DIR / 'manifests/api_responses.json'
assert summary_path.is_file(), summary_path
assert api_manifest_path.is_file(), api_manifest_path
summary = json.loads(summary_path.read_text(encoding='utf-8'))
api_manifest = json.loads(api_manifest_path.read_text(encoding='utf-8'))
assert summary.get('result_id') == 'rq2_frontier_baseline_rev5_v1'
assert summary.get('n_questions') == 100
assert api_manifest.get('completed_result_rows') == 100
display(Markdown((OUTPUT_DIR / 'SUMMARY.md').read_text(encoding='utf-8')))
print('Server-reported model:', api_manifest['response_model'])
print('All logged API usage:', api_manifest['all_logged_usage']['totals'])
print('Parse-retry rows:', api_manifest['parse_retry_rows'])
print('Orphaned API calls:', api_manifest['orphaned_call_count'])

# Batch 5D — Revision 5 Frontier API Baseline

Result identity: `rq2_frontier_baseline_rev5_v1`

All systems use the same 100 frozen Revision 5 questions, ordered evidence windows, ASK policy, and offline verifier. The primary registered comparison is ComplianceGPT versus Gemini 3.5 Flash.

## Primary and supporting outcomes

| System | Strict pass [95% CI] | Full clause coverage | Runtime pass | Clause precision | Clause recall | Mean words |
|---|---:|---:|---:|---:|---:|---:|
| ComplianceGPT, 4-bit selector | 65/100 (0.650) [0.553, 0.736] | 65/100 (0.650) | 100/100 (1.000) | 0.584 | 0.835 | 220.7 |
| Gemini 3.5 Flash, free-form | 66/100 (0.660) [0.563, 0.745] | 67/100 (0.670) | 99/100 (0.990) | 0.696 | 0.775 | 60.2 |
| Qwen2.5-7B, free-form 4-bit | 25/100 (0.250) [0.175, 0.343] | 53/100 (0.530) | 42/100 (0.420) | 0.530 | 0.690 | 45.4 |

## Registered paired strict-pass test

| Comparison | ComplianceGPT only | Gemini only | Both pass | Neither pass | Exact two-sided McNemar p |
|---|---:|---:|---:|---:|---:|
| ComplianceGPT vs Gemini 3.5 Flash | 11 | 12 | 54 | 23 | 1 |

On these 100 paired rows, ComplianceGPT and Gemini 3.5 Flash are not statistically distinguishable on strict pass. The point estimate is not used as evidence of direction.

## Coverage-complete rejections

| System | Coverage-complete | Strict pass within coverage | Lost | Realization loss [95% CI] |
|---|---:|---:|---:|---:|
| ComplianceGPT, 4-bit selector | 65 | 65 | 0 | 0/65 (0.000) [0.000, 0.056] |
| Gemini 3.5 Flash, free-form | 67 | 66 | 1 | 1/67 (0.015) [0.003, 0.080] |
| Qwen2.5-7B, free-form 4-bit | 53 | 25 | 28 | 28/53 (0.528) [0.397, 0.656] |

Two-sided Fisher exact p for ComplianceGPT versus Gemini realization loss: `1`.

ComplianceGPT's observed zero realization loss is predicted by construction; it confirms that the implementation matches the citation-contract specification on the coverage-complete rows.

## ODP status operating point

| System | Sensitivity [95% CI] | Specificity [95% CI] | Status precision [95% CI] | Exact ODP set on positive rows |
|---|---:|---:|---:|---:|
| ComplianceGPT, 4-bit selector | 63/63 (1.000) [0.943, 1.000] | 17/37 (0.459) [0.310, 0.616] | 63/83 (0.759) [0.657, 0.838] | 55/63 (0.873) |
| Gemini 3.5 Flash, free-form | 56/63 (0.889) [0.788, 0.945] | 31/37 (0.838) [0.689, 0.923] | 56/62 (0.903) [0.805, 0.955] | 46/63 (0.730) |
| Qwen2.5-7B, free-form 4-bit | 0/63 (0.000) [0.000, 0.057] | 37/37 (1.000) [0.906, 1.000] | 0/0 (NA) | 21/63 (0.333) |

ODP sensitivity and specificity are reported together. These values use the current author labels and are not independently adjudicated.

## Interpretation boundary

This is a stronger-system baseline, not an isolation of weight precision or architecture. Questions, model-visible evidence, prompt/parser, ODP policy, and verifier are fixed, but the API model and serving runtime differ. The strict-pass comparison measures agreement with the implemented author-labeled rules; it does not establish legal sufficiency or auditor approval. The mechanistic claim concerns realization loss and runtime-verifiable guarantees, not universal accuracy superiority.


Server-reported model: gemini-3.5-flash
All logged API usage: {'cached_content_token_count': 0, 'candidates_token_count': 16813, 'prompt_token_count': 132723, 'thoughts_token_count': 39832, 'tool_use_prompt_token_count': 0, 'total_token_count': 189368}
Parse-retry rows: 0
Orphaned API calls: 0


In [9]:
archive_path = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}.zip'
assert archive_path.is_file(), f'Result ZIP was not created: {archive_path}'
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print('Confirmed result directory:', OUTPUT_DIR)
print('Confirmed result ZIP:', archive_path)
print('Result ZIP SHA-256:', archive_sha256)
print('The execution record and result ZIP are ready for independent audit.')

Confirmed result directory: /content/drive/MyDrive/rq2_frontier_baseline_rev5_v1
Confirmed result ZIP: /content/drive/MyDrive/rq2_frontier_baseline_rev5_v1.zip
Result ZIP SHA-256: 9f088b2ccb410fdd6df8980d693529320fec351e9a01e7400cd7d48fc55dddef
The execution record and result ZIP are ready for independent audit.


## Output artifacts

A completed run writes the result directory and `/content/drive/MyDrive/rq2_frontier_baseline_rev5_v1.zip`. The archive contains the raw API log, contracts, manifests, summary, and integrity hashes needed to reproduce and audit the analysis. Result rows and generated summary values are immutable research records and should not be edited manually.